# EasyVisa Classification Project

## Context

Business communities in the United States are facing high demand for human resources, but one of the constant challenges is identifying and attracting the right talent, which is perhaps the most important element in remaining competitive. Companies in the United States look for hard-working, talented, and qualified individuals both locally as well as abroad.

The Immigration and Nationality Act (INA) of the US permits foreign workers to come to the United States to work on either a temporary or permanent basis. The act also protects US workers against adverse impacts on their wages or working conditions by ensuring US employers' compliance with statutory requirements when they hire foreign workers to fill workforce shortages. The immigration programs are administered by the Office of Foreign Labor Certification (OFLC).

OFLC processes job certification applications for employers seeking to bring foreign workers into the United States and grants certifications in those cases where employers can demonstrate that there are not sufficient US workers available to perform the work at wages that meet or exceed the wage paid for the occupation in the area of intended employment.

## Objective

In FY 2016, the OFLC processed 775,979 employer applications for 1,699,957 positions for temporary and permanent labor certifications. This was a nine percent increase in the overall number of processed applications from the previous year. The process of reviewing every case is becoming a tedious task as the number of applicants is increasing every year.

The increasing number of applicants every year calls for a Machine Learning based solution that can help in shortlisting the candidates having higher chances of VISA approval. OFLC has hired the firm EasyVisa for data-driven solutions. You as a data scientist at EasyVisa have to analyze the data provided and, with the help of a classification model:

Facilitate the process of visa approvals.
Recommend a suitable profile for the applicants for whom the visa should be certified or denied based on the drivers that significantly influence the case status.

In [1]:
!pip install imbalanced-learn==0.13.0 -q


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# To undersample and oversample the data
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# To suppress warnings
import warnings
warnings.filterwarnings("ignore")


In [3]:
# Load the dataset
df = pd.read_csv('C:/Users/yadukrishna.prithvin/OneDrive - The Hackett Group, Inc/AI ML Course/Advanced ML/Projec/EasyVisa.csv')
df.head()


case_id


continent


education_of_employee


has_job_experience


requires_job_training


no_of_employees


yr_of_estab


region_of_employment


prevailing_wage


unit_of_wage


full_time_position


case_status










0


EZYV01


Asia


High School


N


N


14513


2007


West


592.2029


Hour


Y


Denied






1


EZYV02


Asia


Master's


Y


N


2412


2002


Northeast


83425.6500


Year


Y


Certified






2


EZYV03


Asia


Bachelor's


N


Y


44444


2008


West


122996.8600


Year


Y


Denied






3


EZYV04


Asia


Bachelor's


N


N


98


1897


West


83434.0300


Year


Y


Denied






4


EZYV05


Africa


Master's


Y


N


1082


2005


South


149907.3900


Year


Y


Certified

In [4]:
df.shape


(25480, 12)

In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25480 entries, 0 to 25479
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   case_id                25480 non-null  object 
 1   continent              25480 non-null  object 
 2   education_of_employee  25480 non-null  object 
 3   has_job_experience     25480 non-null  object 
 4   requires_job_training  25480 non-null  object 
 5   no_of_employees        25480 non-null  int64  
 6   yr_of_estab            25480 non-null  int64  
 7   region_of_employment   25480 non-null  object 
 8   prevailing_wage        25480 non-null  float64
 9   unit_of_wage           25480 non-null  object 
 10  full_time_position     25480 non-null  object 
 11  case_status            25480 non-null  object 
dtypes: float64(1), int64(2), object(9)
memory usage: 2.3+ MB

In [6]:
df['case_status'].value_counts()


case_status
Certified    17018
Denied        8462
Name: count, dtype: int64

In [7]:
df.describe()


no_of_employees


yr_of_estab


prevailing_wage










count


25480.000000


25480.000000


25480.000000






mean


5667.043210


1979.409929


74455.814592






std


22877.928848


42.366929


52815.942327






min


-26.000000


1800.000000


2.136700






25%


1022.000000


1976.000000


34015.480000






50%


2109.000000


1997.000000


70308.210000






75%


3504.000000


2005.000000


107735.512500






max


602069.000000


2016.000000


319210.270000

In [8]:
# Drop case_id and handle missing values
df.drop(columns=['case_id'], inplace=True)
df.dropna(inplace=True)


In [9]:
# checking missing values in the data
df.isna().sum()


continent                0
education_of_employee    0
has_job_experience       0
requires_job_training    0
no_of_employees          0
yr_of_estab              0
region_of_employment     0
prevailing_wage          0
unit_of_wage             0
full_time_position       0
case_status              0
dtype: int64

In [10]:
df["continent"] = df["continent"].astype("category")
df["education_of_employee"] = df["education_of_employee"].astype("category")
df["has_job_experience"] = df["has_job_experience"].astype("category")
df["requires_job_training"] = df["requires_job_training"].astype("category")
df["region_of_employment"] = df["region_of_employment"].astype("category")
df["full_time_position"] = df["full_time_position"].astype("category")
df["case_status"] = df["case_status"].astype("category")


In [11]:
df["case_status"].value_counts(1)


case_status
Certified    0.667896
Denied       0.332104
Name: proportion, dtype: float64

In [12]:
# Convert wage to yearly
wage_conversion = {'Hour': 2080, 'Week': 52, 'Month': 12, 'Year': 1}
df['wage_yearly'] = df['prevailing_wage'] * df['unit_of_wage'].map(wage_conversion)
df.drop(columns=['prevailing_wage', 'unit_of_wage'], inplace=True)

# Create company age feature
df['company_age'] = 2016 - df['yr_of_estab']
df.drop(columns=['yr_of_estab'], inplace=True)

# Encode categorical variables
label_cols = ['continent', 'education_of_employee', 'has_job_experience', 'requires_job_training', 'region_of_employment', 'full_time_position']
le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col])

# Encode target variable
df['case_status'] = df['case_status'].map({'Certified': 1, 'Denied': 0})


In [13]:
X = df.drop('case_status', axis=1)
y = df['case_status']

# Split into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


## OverSampling

In [14]:
# Fit SMOTE on train data(Synthetic Minority Oversampling Technique)
sm = SMOTE(sampling_strategy=0.6, k_neighbors=5, random_state=1)
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)


In [15]:
print("Before OverSampling, count of label '1': {}".format(sum(y_train == 1)))
print("Before OverSampling, count of label '0': {} \n".format(sum(y_train == 0)))

print("After OverSampling, count of label '1': {}".format(sum(y_train_over == 1)))
print("After OverSampling, count of label '0': {} \n".format(sum(y_train_over == 0)))

print("After OverSampling, the shape of train_X: {}".format(X_train_over.shape))
print("After OverSampling, the shape of train_y: {} \n".format(y_train_over.shape))


Before OverSampling, count of label '1': 11899
Before OverSampling, count of label '0': 5937 

After OverSampling, count of label '1': 11899
After OverSampling, count of label '0': 7139 

After OverSampling, the shape of train_X: (19038, 9)
After OverSampling, the shape of train_y: (19038,)

#### Let's train a decision tree classifier using the oversampled data

In [16]:
dtree1 = DecisionTreeClassifier(random_state=1, max_depth=4)

# training the decision tree model with oversampled training set
dtree1.fit(X_train_over, y_train_over)


DecisionTreeClassifier(max_depth=4, random_state=1)
In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook. 
On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.
DecisionTreeClassifier
?
Documentation for DecisionTreeClassifier
i
Fitted
DecisionTreeClassifier(max_depth=4, random_state=1)

#### We have trained the model, Let's check the performance on the oversampled train and validation sets

In [17]:
# Predicting the target for train and validation set
pred_train = dtree1.predict(X_train_over)
pred_val = dtree1.predict(X_val)


In [18]:
# Checking recall score on oversampled train and validation set
print(recall_score(y_train_over, pred_train))
print(recall_score(y_val, pred_val))


0.9014202874191108
0.8928571428571429

In [19]:
# Checking accuracy score on oversampled train and validation set
print(accuracy_score(y_train_over, pred_train))
print(accuracy_score(y_val, pred_val))


0.7108940014707428
0.7247514390371533

#### After checking the performance, let's look at the confusion matrices of the train and validation data

In [20]:
# Confusion matrix for oversampled train data
cm = confusion_matrix(y_train_over, pred_train)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="g")
plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")


Text(58.222222222222214, 0.5, 'Actual Values')

In [21]:
# Confusion matrix for validation data
cm = confusion_matrix(y_val, pred_val)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="g")
plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")
plt.show()


## Boosting with Over Sampling

In [22]:
#Train AdaBoosting
ada_ov = AdaBoostClassifier()
ada_ov.fit(X_train_over, y_train_over)

# Predictions
y_train_pred = ada_ov.predict(X_train_over)
y_val_pred = ada_ov.predict(X_val)
y_test_pred = ada_ov.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train_over, y_train_pred)
print_scores("Validation", y_val, y_val_pred)
print_scores("Test", y_test, y_test_pred)


Train Accuracy: 0.7192
Train Precision: 0.7368
Train Recall: 0.8568
Validation Accuracy: 0.7271
Validation Precision: 0.7651
Validation Recall: 0.8587
Test Accuracy: 0.7250
Test Precision: 0.7592
Test Recall: 0.8592

In [23]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_pred, 'AdaBoost Confusion Matrix - Test')


In [24]:
# AdaBoost Feature importance
importances = ada_ov.feature_importances_
features = X.columns
sns.barplot(x=importances, y=features)
plt.title('AdaBoost Feature Importances')
plt.show()


Here the top three important features are Education of employee, Region of employement and yearly wage

In [25]:
# Train Gradient Boosting
gb_ov = GradientBoostingClassifier()
gb_ov.fit(X_train_over, y_train_over)

# Predictions
y_train_predi = gb_ov.predict(X_train_over)
y_val_predi = gb_ov.predict(X_val)
y_test_predi = gb_ov.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train_over, y_train_predi)
print_scores("Validation", y_val, y_val_predi)
print_scores("Test", y_test, y_test_predi)


Train Accuracy: 0.7389
Train Precision: 0.7601
Train Recall: 0.8508
Validation Accuracy: 0.7407
Validation Precision: 0.7813
Validation Recall: 0.8544
Test Accuracy: 0.7347
Test Precision: 0.7773
Test Recall: 0.8427

In [26]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predi, 'Gradient Boost Confusion Matrix - Test')


In [27]:
# Gradient Boost Feature importance
importances = gb_ov.feature_importances_
features = X.columns
sns.barplot(x=importances, y=features)
plt.title('Gradient Boost Feature Importances')
plt.show()


According to this boosting, the Three main important features can be seen as Education of employee, Job experience and yearly Wage

In [28]:
# AdaBoost tuning
ada_params = {'n_estimators': [50, 100, 200], 'learning_rate': [0.5, 1.0, 1.5]}
ada_gridov = GridSearchCV(AdaBoostClassifier(), ada_params, cv=3, scoring='f1')
ada_gridov.fit(X_train_over, y_train_over)

# Predictions
y_train_predit = ada_gridov.predict(X_train_over)
y_val_predit = ada_gridov.predict(X_val)
y_test_predit = ada_gridov.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train_over, y_train_predit)
print_scores("Validation", y_val, y_val_predit)
print_scores("Test", y_test, y_test_predit)


Train Accuracy: 0.7217
Train Precision: 0.7410
Train Recall: 0.8528
Validation Accuracy: 0.7318
Validation Precision: 0.7703
Validation Recall: 0.8579
Test Accuracy: 0.7287
Test Precision: 0.7644
Test Recall: 0.8561

In [29]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predit, 'Tuned AdaBoost Confusion Matrix - Test')


In [30]:
# Tuned AdBoost Feature importance
importance = ada_gridov.best_estimator_.feature_importances_
features = X.columns
sns.barplot(x=importance, y=features)
plt.title('Tuned AdaBoost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()


According to this boosting, the three main importance features can be seen as Education of employee, Region of employment and yearly wage

In [31]:
# Gradient Boosting tuning
gb_params = {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}
gb_gridov = GridSearchCV(GradientBoostingClassifier(), gb_params, cv=3, scoring='f1')
gb_gridov.fit(X_train_over, y_train_over)

# Predictions
y_train_predict = gb_gridov.predict(X_train_over)
y_val_predict = gb_gridov.predict(X_val)
y_test_predict = gb_gridov.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train_over, y_train_predict)
print_scores("Validation", y_val, y_val_predict)
print_scores("Test", y_test, y_test_predict)


Train Accuracy: 0.7515
Train Precision: 0.7729
Train Recall: 0.8533
Validation Accuracy: 0.7418
Validation Precision: 0.7849
Validation Recall: 0.8498
Test Accuracy: 0.7339
Test Precision: 0.7801
Test Recall: 0.8356

In [32]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predict, 'Tuned Gradient Boost Confusion Matrix - Test')


In [33]:
# Tuned Gradient Boost Feature importance
importance = gb_gridov.best_estimator_.feature_importances_
features = X.columns
sns.barplot(x=importance, y=features)
plt.title('Tuned Gradient Boost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()


According to this boosting, the three main feature importance are Education of employee, Job Experience and Yearly wage.

## Undersampling train data using Random Undersampler

In [34]:
# fit random under sampler on the train data
rus = RandomUnderSampler(random_state=1, sampling_strategy = 1)
X_train_un, y_train_un = rus.fit_resample(X_train, y_train)


In [35]:
print("Before Under Sampling, count of label '1': {}".format(sum(y_train == 1)))
print("Before Under Sampling, count of label '0': {} \n".format(sum(y_train == 0)))

print("After Under Sampling, count of label '1': {}".format(sum(y_train_un == 1)))
print("After Under Sampling, count of label '0': {} \n".format(sum(y_train_un == 0)))

print("After Under Sampling, the shape of train_X: {}".format(X_train_un.shape))
print("After Under Sampling, the shape of train_y: {} \n".format(y_train_un.shape))


Before Under Sampling, count of label '1': 11899
Before Under Sampling, count of label '0': 5937 

After Under Sampling, count of label '1': 5937
After Under Sampling, count of label '0': 5937 

After Under Sampling, the shape of train_X: (11874, 9)
After Under Sampling, the shape of train_y: (11874,)

#### Let's train a decision tree classifier using the undersampled data

In [36]:
dtree2 = DecisionTreeClassifier(random_state=1, max_depth=4)

# training the decision tree model with oversampled training set
dtree2.fit(X_train_un, y_train_un)


DecisionTreeClassifier(max_depth=4, random_state=1)
In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook. 
On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.
DecisionTreeClassifier
?
Documentation for DecisionTreeClassifier
i
Fitted
DecisionTreeClassifier(max_depth=4, random_state=1)

#### We have trained the model, let's check the performance on undersampled train and validation data

In [37]:
# Predicting the target for train and validation set
pred_train = dtree2.predict(X_train_un)
pred_val = dtree2.predict(X_val)


In [38]:
# Checking recall score on oversampled train and validation set
print(recall_score(y_train_un, pred_train))
print(recall_score(y_val, pred_val))


0.6912582112177867
0.6948757763975155

In [39]:
# Checking accuracy score on undersampled train and validation set
print(accuracy_score(y_train_un, pred_train))
print(accuracy_score(y_val, pred_val))


0.6790466565605525
0.6789638932496075

#### After checking the performance, let's look at the confusion matrices of the train and validation data

In [40]:
# Confusion matrix for undersampled train data
cm = confusion_matrix(y_train_un, pred_train)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="g")
plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")
plt.show()


In [41]:
# Confusion matrix for validation data
cm = confusion_matrix(y_val, pred_val)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="g")
plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")
plt.show()


### let's now check the performance of dtree2 using the test data

In [42]:
# Now we have identified the best model, let's check its performance on test set
print(recall_score(y_test, dtree2.predict(X_test)))

cm = confusion_matrix(y_test, dtree2.predict(X_test))
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="g")
plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")
plt.show()


0.6917027133307118

## Boosting with Under Sampling

In [43]:
#Train AdaBoosting
ada_un = AdaBoostClassifier()
ada_un.fit(X_train_un, y_train_un)

# Predictions
y_train_pred = ada_un.predict(X_train_un)
y_val_pred = ada_un.predict(X_val)
y_test_pred = ada_un.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train_un, y_train_pred)
print_scores("Validation", y_val, y_val_pred)
print_scores("Test", y_test, y_test_pred)


Train Accuracy: 0.6929
Train Precision: 0.6737
Train Recall: 0.7482
Validation Accuracy: 0.7080
Validation Precision: 0.8047
Validation Recall: 0.7484
Test Accuracy: 0.7098
Test Precision: 0.8033
Test Recall: 0.7468

In [44]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_pred, 'AdaBoost Confusion Matrix - Test')


In [45]:
# AdaBoost Feature importance
importances = ada_un.feature_importances_
features = X.columns
sns.barplot(x=importances, y=features)
plt.title('AdaBoost Feature Importances')
plt.show()


Here, the three main features can be seen as Education of Employee, Region of Employment and yearly Wage

In [46]:
# Train Gradient Boosting
gbun = GradientBoostingClassifier()
gbun.fit(X_train_un, y_train_un)

# Predictions
y_train_predi = gbun.predict(X_train_un)
y_val_predi = gbun.predict(X_val)
y_test_predi = gbun.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train_un, y_train_predi)
print_scores("Validation", y_val, y_val_predi)
print_scores("Test", y_test, y_test_predi)


Train Accuracy: 0.7214
Train Precision: 0.7170
Train Recall: 0.7315
Validation Accuracy: 0.7077
Validation Precision: 0.8255
Validation Recall: 0.7182
Test Accuracy: 0.7145
Test Precision: 0.8321
Test Recall: 0.7153

In [47]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predi, 'Gradient Boost Confusion Matrix - Test')


In [48]:
# Gradient Boost Feature importance
importances = gbun.feature_importances_
features = X.columns
sns.barplot(x=importances, y=features)
plt.title('Gradient Boost Feature Importances')
plt.show()


According to this boosting, the main three features can be seen as Education of employee, JOb experience and Yearly wage

In [49]:
# AdaBoost tuning
ada_params = {'n_estimators': [50, 100, 200], 'learning_rate': [0.5, 1.0, 1.5]}
ada_gridun = GridSearchCV(AdaBoostClassifier(), ada_params, cv=3, scoring='f1')
ada_gridun.fit(X_train_un, y_train_un)

# Predictions
y_train_predit = ada_gridun.predict(X_train_un)
y_val_predit = ada_gridun.predict(X_val)
y_test_predit = ada_gridun.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train_un, y_train_predit)
print_scores("Validation", y_val, y_val_predit)
print_scores("Test", y_test, y_test_predit)


Train Accuracy: 0.6929
Train Precision: 0.6737
Train Recall: 0.7482
Validation Accuracy: 0.7080
Validation Precision: 0.8047
Validation Recall: 0.7484
Test Accuracy: 0.7098
Test Precision: 0.8033
Test Recall: 0.7468

In [50]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predit, 'Tuned AdaBoost Confusion Matrix - Test')


In [51]:
# Tuned AdBoost Feature importance
importance = ada_gridun.best_estimator_.feature_importances_
features = X.columns
sns.barplot(x=importance, y=features)
plt.title('Tuned AdaBoost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()


Here, the main three important features can be seen as Education of employees, region of employment, and yearly wage

In [52]:
# Gradient Boosting tuning
gb_params = {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}
gb_gridun = GridSearchCV(GradientBoostingClassifier(), gb_params, cv=3, scoring='f1')
gb_gridun.fit(X_train_un, y_train_un)

# Predictions
y_train_predict = gb_gridun.predict(X_train_un)
y_val_predict = gb_gridun.predict(X_val)
y_test_predict = gb_gridun.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train_un, y_train_predict)
print_scores("Validation", y_val, y_val_predict)
print_scores("Test", y_test, y_test_predict)


Train Accuracy: 0.7392
Train Precision: 0.7242
Train Recall: 0.7726
Validation Accuracy: 0.7085
Validation Precision: 0.8165
Validation Recall: 0.7321
Test Accuracy: 0.7166
Test Precision: 0.8205
Test Recall: 0.7350

In [53]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predict, 'Tuned Gradient Boost Confusion Matrix - Test')


In [54]:
# Tuned Gradient Boost Feature importance
importance = ada_gridun.best_estimator_.feature_importances_
features = X.columns
sns.barplot(x=importance, y=features)
plt.title('Tuned Gradient Boost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()


Here, the three main importance can be seen as education of employee, Region of employment and yearly wage

## Boosting without Sampling

In [55]:
#Train AdaBoosting
ada = AdaBoostClassifier()
ada.fit(X_train, y_train)

# Predictions
y_train_pred = ada.predict(X_train)
y_val_pred = ada.predict(X_val)
y_test_pred = ada.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train, y_train_pred)
print_scores("Validation", y_val, y_val_pred)
print_scores("Test", y_test, y_test_pred)


Train Accuracy: 0.7249
Train Precision: 0.7412
Train Recall: 0.9030
Validation Accuracy: 0.7237
Validation Precision: 0.7417
Validation Recall: 0.9053
Test Accuracy: 0.7248
Test Precision: 0.7384
Test Recall: 0.9080

In [56]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_pred, 'AdaBoost Confusion Matrix - Test')


In [57]:
# AdaBoost Feature importance
importances = ada.feature_importances_
features = X.columns
sns.barplot(x=importances, y=features)
plt.title('AdaBoost Feature Importances')
plt.show()


Here the main three important features are Education, Region of employement and Continent/yearly wage.

In [58]:
# Train Gradient Boosting
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

# Predictions
y_train_predi = gb.predict(X_train)
y_val_predi = gb.predict(X_val)
y_test_predi = gb.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train, y_train_predi)
print_scores("Validation", y_val, y_val_predi)
print_scores("Test", y_test, y_test_predi)


Train Accuracy: 0.7528
Train Precision: 0.7765
Train Recall: 0.8838
Validation Accuracy: 0.7446
Validation Precision: 0.7723
Validation Recall: 0.8808
Test Accuracy: 0.7357
Test Precision: 0.7637
Test Recall: 0.8730

In [59]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predi, 'Gradient Boost Confusion Matrix - Test')


In [60]:
# Gradient Boost Feature importance
importances = gb.feature_importances_
features = X.columns
sns.barplot(x=importances, y=features)
plt.title('Gradient Boost Feature Importances')
plt.show()


Here the important features are Education of employee, job experience and yearly wage

In [61]:
# AdaBoost tuning
ada_params = {'n_estimators': [50, 100, 200], 'learning_rate': [0.5, 1.0, 1.5]}
ada_grid = GridSearchCV(AdaBoostClassifier(), ada_params, cv=3, scoring='f1')
ada_grid.fit(X_train, y_train)
y_pred_gb = ada_grid.predict(X_test)

# Predictions
y_train_predit = ada_grid.predict(X_train)
y_val_predit = ada_grid.predict(X_val)
y_test_predit = ada_grid.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train, y_train_predit)
print_scores("Validation", y_val, y_val_predit)
print_scores("Test", y_test, y_test_predit)


Train Accuracy: 0.7245
Train Precision: 0.7397
Train Recall: 0.9058
Validation Accuracy: 0.7200
Validation Precision: 0.7381
Validation Recall: 0.9061
Test Accuracy: 0.7234
Test Precision: 0.7363
Test Recall: 0.9103

In [62]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predit, 'Tuned AdaBoost Confusion Matrix - Test')


In [63]:
# Tuned AdBoost Feature importance
importance = ada_grid.best_estimator_.feature_importances_
features = X.columns
sns.barplot(x=importance, y=features)
plt.title('Tuned AdaBoost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()


Here the importance features are Education of employee, region of employement and yearly wage

In [64]:
# Gradient Boosting tuning
gb_params = {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}
gb_grid = GridSearchCV(GradientBoostingClassifier(), gb_params, cv=3, scoring='f1')
gb_grid.fit(X_train, y_train)
y_pred_gb = gb_grid.predict(X_test)

# Predictions
y_train_predict = gb_grid.predict(X_train)
y_val_predict = gb_grid.predict(X_val)
y_test_predict = gb_grid.predict(X_test)

# Scores
def print_scores(name, y_true, y_pred):
    print(f"{name} Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"{name} Recall: {recall_score(y_true, y_pred):.4f}")

print_scores("Train", y_train, y_train_predict)
print_scores("Validation", y_val, y_val_predict)
print_scores("Test", y_test, y_test_predict)


Train Accuracy: 0.7517
Train Precision: 0.7762
Train Recall: 0.8821
Validation Accuracy: 0.7439
Validation Precision: 0.7726
Validation Recall: 0.8785
Test Accuracy: 0.7378
Test Precision: 0.7667
Test Recall: 0.8710

In [65]:
# Confusion matrix heatmap
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
plot_confusion(y_test, y_test_predict, 'Tuned Gradient Boost Confusion Matrix - Test')


In [66]:
# Tuned Gradient Boost Feature importance
importance = ada_grid.best_estimator_.feature_importances_
features = X.columns
sns.barplot(x=importance, y=features)
plt.title('Tuned Gradient Boost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()


Here, the importance features are education of employee, REgion of employement and yearly wage

In [67]:
 ##  Function to calculate different metric scores of the model - Accuracy, Recall and Precision
def get_metrics_score(model,flag=True):
    '''
    model : classifier to predict values of X

    '''
    # defining an empty list to store train and test results
    score_list=[]

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    train_acc = model.score(X_train,y_train)
    test_acc = model.score(X_test,y_test)

    train_recall = metrics.recall_score(y_train,pred_train)
    test_recall = metrics.recall_score(y_test,pred_test)

    train_precision = metrics.precision_score(y_train,pred_train)
    test_precision = metrics.precision_score(y_test,pred_test)

    score_list.extend((train_acc,test_acc,train_recall,test_recall,train_precision,test_precision))

    # If the flag is set to True then only the following print statements will be dispayed. The default value is set to True.
    if flag == True:
        print("Accuracy on training set : ",model.score(X_train,y_train))
        print("Accuracy on test set : ",model.score(X_test,y_test))
        print("Recall on training set : ",metrics.recall_score(y_train,pred_train))
        print("Recall on test set : ",metrics.recall_score(y_test,pred_test))
        print("Precision on training set : ",metrics.precision_score(y_train,pred_train))
        print("Precision on test set : ",metrics.precision_score(y_test,pred_test))

    return score_list # returning the list with train and test scores


In [68]:
#defining list of models
models = [ada, ada_grid, ada_ov, ada_gridov, ada_un, ada_gridun, gb, gb_grid, gb_ov, gb_gridov, gbun, gb_gridun]

# defining empty lists to add train and test results
acc_train = []
acc_test = []
recall_train = []
recall_test = []
precision_train = []
precision_test = []

# looping through all the models to get the accuracy, precall and precision scores
for model in models:
    j = get_metrics_score(model,False)
    acc_train.append(np.round(j[0],2))
    acc_test.append(np.round(j[1],2))
    recall_train.append(np.round(j[2],2))
    recall_test.append(np.round(j[3],2))
    precision_train.append(np.round(j[4],2))
    precision_test.append(np.round(j[5],2))


In [69]:
comparison_frame = pd.DataFrame({'Model':['AdaBoost',' AdaBoost Tuned',' AdaBoost Oversampled Data', 'AdaBoost Over Sampled Data and Tuned', 'Ada Boost Under Sampled Data', 
                                          'Ada Boost Under Sampled Data and Tuned','Gradient Boost',
                                          'Gradient Boost Tuned','Gradient Boost Oversampled Data', 'Gradient Boost Over Sampled Data and Tuned', 'Gradient Boost Under Sampled Data',
                                         'Ada Boost Under Sampled Data and Tuned'],
                                          'Train_Accuracy': acc_train,'Test_Accuracy': acc_test,
                                          'Train_Recall':recall_train,'Test_Recall':recall_test,
                                          'Train_Precision':precision_train,'Test_Precision':precision_test})
comparison_frame


Model


Train_Accuracy


Test_Accuracy


Train_Recall


Test_Recall


Train_Precision


Test_Precision










0


AdaBoost


0.72


0.72


0.90


0.91


0.74


0.74






1


AdaBoost Tuned


0.81


0.81


0.91


0.91


0.74


0.74






2


AdaBoost Oversampled Data


0.73


0.73


0.86


0.86


0.76


0.76






3


AdaBoost Over Sampled Data and Tuned


0.81


0.81


0.85


0.86


0.77


0.76






4


Ada Boost Under Sampled Data


0.71


0.71


0.75


0.75


0.81


0.80






5


Ada Boost Under Sampled Data and Tuned


0.78


0.77


0.75


0.75


0.81


0.80






6


Gradient Boost


0.75


0.74


0.88


0.87


0.78


0.76






7


Gradient Boost Tuned


0.83


0.82


0.88


0.87


0.78


0.77






8


Gradient Boost Oversampled Data


0.75


0.73


0.85


0.84


0.79


0.78






9


Gradient Boost Over Sampled Data and Tuned


0.82


0.81


0.85


0.84


0.80


0.78






10


Gradient Boost Under Sampled Data


0.72


0.71


0.72


0.72


0.83


0.83






11


Ada Boost

### Final Choice

Gradient Boost Tuned is the best model with 0.82 accuracy, 0.77 precision, and 0.87 recall, offering balanced performance and strong generalization for automating visa screening, hence I would choose this model.

### Other take aways

Normal Boosting provides good accuracy (AdaBoost: 0.81, Gradient Boost: 0.82) but tends to favor the majority class.
Oversampling improves minority class precision and recall (e.g., Gradient Boost Oversampled reaches 0.78 precision), while Undersampling increases precision (AdaBoost: 0.82) but reduces overall accuracy due to loss of majority class data.

### Business Recommendations

#### Deploy the Tuned Gradient Boosting Model:

Use the Gradient Boosting model (with hyperparameter tuning) for automated initial screening of visa applications, as it provides the highest accuracy (0.82), precision (0.77), and recall (0.87), ensuring balanced and reliable predictions for visa approvals.

#### Prioritize High-Probability Applications:

Leverage model outputs to flag applications with a high likelihood of approval for expedited review, while directing low-probability cases for further manual scrutiny. This will streamline the OFLC’s review process and reduce manual workload.

#### Focus on Key Approval Drivers:

Guide employers and applicants to strengthen applications by emphasizing key factors such as higher education, prior job experience, larger company size, competitive wages, and full-time positions, as these significantly influence visa certification outcomes.

#### Monitor and Improve Model Performance:

Regularly retrain and monitor the model with new data to maintain accuracy and fairness. Consider integrating oversampling and undersampling techniques to further improve model fairness, especially if class imbalance is detected.

#### Share Insights for Better Applications:

Communicate actionable insights to employers and applicants, helping them understand what drives successful visa approvals and enabling them to prepare stronger applications.